<a href="https://colab.research.google.com/github/MiguelAP6/IAAC/blob/Pedro-Ferreira/Deployment%2C_Artefactos_%26_Teste_PF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from google.colab import files

# 1. Garantir a existência da pasta 'src' e do módulo 'data_processing.py' (M)
if not os.path.exists("src/data_processing.py"):
    os.makedirs("src", exist_ok=True)
    code_src = """import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

def load_and_clean_data(df_input, target_col='Threat Level'):
    df = df_input.dropna().copy()
    id_cols = [c for c in df.columns if 'id' in c.lower() or 'date' in c.lower() or 'timestamp' in c.lower()]
    if id_cols:
        df = df.drop(columns=id_cols)
    return df

def prepare_pipeline(df, target_col='Threat Level'):
    if target_col not in df.columns:
        target_col = df.columns[-1]

    X = df.drop(columns=[target_col])
    y = df[target_col]

    label_encoders = {}
    for col in X.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

    target_encoder = LabelEncoder()
    y_encoded = target_encoder.fit_transform(y.astype(str))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return {
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test,
        'scaler': scaler,
        'target_encoder': target_encoder,
        'label_encoders': label_encoders,
        'feature_names': list(X.columns)
    }
"""
    with open("src/data_processing.py", "w") as f:
        f.write(code_src)

# 2. Carregar e preparar os dados caso não estejam na memória (M)
if 'X_train_scaled' not in globals() or 'scaler' not in globals():
    from src.data_processing import load_and_clean_data, prepare_pipeline

    if 'df' not in globals():
        print("Dataset não encontrado em memória. Por favor, faz o upload do ficheiro CSV:")
        uploaded = files.upload()
        file_name = list(uploaded.keys())[0]
        df = pd.read_csv(file_name)

    df_clean = load_and_clean_data(df)
    prep_data = prepare_pipeline(df_clean)

    X_train_scaled = prep_data['X_train_scaled']
    X_test_scaled = prep_data['X_test_scaled']
    y_train = prep_data['y_train']
    y_test = prep_data['y_test']
    scaler = prep_data['scaler']
    target_encoder = prep_data['target_encoder']
    label_encoders = prep_data['label_encoders']
    feature_names = prep_data['feature_names']

# 3. Treinar/obter o modelo final (M)
if 'best_tuned_model' not in globals() and 'best_base_model' not in globals():
    from sklearn.ensemble import RandomForestClassifier
    print("A treinar o modelo de classificação final (Random Forest)...")
    final_model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, class_weight='balanced')
    final_model.fit(X_train_scaled, y_train)
else:
    final_model = best_tuned_model if 'best_tuned_model' in globals() else best_base_model

# 4. Guardar Artefactos na pasta 'models/' (M)
os.makedirs("models", exist_ok=True)
model_path = "models/cybersecurity_threats_evaluated_model.joblib"

artifacts = {
    "model": final_model,
    "scaler": scaler,
    "target_encoder": target_encoder,
    "label_encoders": label_encoders if 'label_encoders' in globals() else {},
    "feature_names": feature_names if 'feature_names' in globals() else []
}

joblib.dump(artifacts, model_path)
print(f"=== PARTE 5: ARTEFACTO GUARDADO COM SUCESSO EM '{model_path}' (M) ===")

# 5. Teste de Inferência / Recarregamento do Modelo
print("\n=== TESTE DE INFERÊNCIA E CARREGAMENTO ===")

loaded_artifacts = joblib.load(model_path)
model_loaded = loaded_artifacts["model"]
scaler_loaded = loaded_artifacts["scaler"]
target_enc_loaded = loaded_artifacts["target_encoder"]

sample_data = X_test_scaled[0].reshape(1, -1)

sample_pred_encoded = model_loaded.predict(sample_data)[0]
sample_pred_class = target_enc_loaded.inverse_transform([sample_pred_encoded])[0]

if hasattr(model_loaded, "predict_proba"):
    sample_proba = model_loaded.predict_proba(sample_data).max() * 100
    print(f"Previsão do Modelo: Ameaça do Tipo '{sample_pred_class}' (Confiança: {sample_proba:.1f}%)")
else:
    print(f"Previsão do Modelo: Ameaça do Tipo '{sample_pred_class}'")

print("\nPipeline de Machine Learning concluído com sucesso!")

Dataset não encontrado em memória. Por favor, faz o upload do ficheiro CSV:


Saving archive.zip to archive.zip
A treinar o modelo de classificação final (Random Forest)...
=== PARTE 5: ARTEFACTO GUARDADO COM SUCESSO EM 'models/cybersecurity_threats_evaluated_model.joblib' (M) ===

=== TESTE DE INFERÊNCIA E CARREGAMENTO ===
Previsão do Modelo: Ameaça do Tipo 'Encryption' (Confiança: 27.8%)

Pipeline de Machine Learning concluído com sucesso!
